In [4]:
from FultonMarket.FultonMarket import FultonMarket as FM
from copy import deepcopy

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



jax:    0.5.2
jaxlib: 0.5.2
numpy:  2.4.2
python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
device info: GRID A100X-10C-1, 1 local devices"
process_count: 1
platform: uname_result(system='Linux', node='especially-major-fawn', release='6.17.0-14-generic', version='#14~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jan 15 15:52:10 UTC 2', machine='x86_64')


$ nvidia-smi
Sun Mar  8 22:15:45 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|==========================

In [5]:
input_dir = '/media/volume/Large-Storage/kor/Luis_RS_cmpds/'
name = 'RS1125'
output_dir = '/media/volume/Large-Storage/kor/RS1125_output/'
replicate = '0'

total_sim_time = 20
sub_sim_length = 2
iter_length = 0.005

In [6]:
# Inputs
import os
input_sys = os.path.join(input_dir, name+'_sys.xml')
input_state = os.path.join(input_dir, name+'_state.xml')
input_pdb = os.path.join(input_dir, name+'.pdb')
output_dir = os.path.join(output_dir, name + '_' + str(replicate))
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
assert os.path.exists(output_dir)

In [ ]:
# Run rep exchange
market = FM(input_pdb=input_pdb, input_system=input_sys, input_state=input_state, 
            T_min=310, T_max=325, n_replicates=20, sele_str=None)


# try:
market.run(iter_length=iter_length, dt=2.0, sim_length=sub_sim_length,
           convergence_thresh=None, resSeqs=None,
           total_sim_time=total_sim_time, output_dir=output_dir)

03/08/2026 22:15:49//Welcome to FultonMarket.
03/08/2026 22:15:51//Found input_pdb: /media/volume/Large-Storage/kor/Luis_RS_cmpds/RS1125.pdb
03/08/2026 22:15:52//Found input_system: /media/volume/Large-Storage/kor/Luis_RS_cmpds/RS1125_sys.xml
03/08/2026 22:15:53//Found input_state: /media/volume/Large-Storage/kor/Luis_RS_cmpds/RS1125_state.xml
03/08/2026 22:15:53//Found total simulation time of 20 nanoseconds
03/08/2026 22:15:53//Found iteration length of 0.005 nanoseconds
03/08/2026 22:15:53//Found timestep of 2.0 femtoseconds
03/08/2026 22:15:53//Found length of simulation to be 2 nanoseconds
03/08/2026 22:15:53//Found number of replicates 20
03/08/2026 22:15:53//Found initial acceptance rate threshold 0.5
03/08/2026 22:15:53//Found terminal acceptance rate threshold 0.35
03/08/2026 22:15:53//Found output_dir /media/volume/Large-Storage/kor/RS1125_output/RS1125_0
03/08/2026 22:15:53//Found Temperature Schedule [np.float64(310.0), np.float64(310.5), np.float64(311.0), np.float64(311.5

03/08/2026 22:15:54//Removing /media/volume/Large-Storage/kor/RS1125_output/RS1125_0/output.ncdf


03/08/2026 22:15:59//CYCLE 0 advancing 5.0 iterations


#### NOT THIS
```
#Try the easiest thing first, load the system and double the masses
mass_scale = 2


from openmm import *
from openmm.app import *
from openmm.unit import *

with open(input_sys, 'r') as f:
    system = XmlSerializer.deserialize(f.read())

unique_masses = []
for i in range(system.getNumParticles()):
    mass = system.getParticleMass(i)
    if mass < 1.1*dalton:
        system.setParticleMass(i, mass_scale*mass)

with open(os.path.join(input_dir, name+'_sys_D.xml'), 'w') as f:
    f.write(XmlSerializer.serialize(system))

input_sys = os.path.join(input_dir, name+'_sys_D.xml')
```

### Perform Hydrogen Mass Augmentation by 'Stealing' from the Bonded Heavy Atom (up to 4 Dalton total H mass)

In [4]:
from openmm import *
from openmm.app import *
from openmm.unit import *
import numpy as np


pdb = PDBFile(input_pdb)
with open(input_sys, 'r') as f:
    normal_sys = XmlSerializer.deserialize(f.read())
top = pdb.topology
top.getNumBonds()

35618

In [5]:


for i in np.linspace(1, 3.5, 40):
    print('#################################################################################################################')
    print(f'#################################{i=}##########################################################')
    print('#################################################################################################################')

    sys = deepcopy(normal_sys)
    
    delta_H = i*element.hydrogen.mass
    
    print('Expect Masses')
    for el in [element.hydrogen, element.carbon, element.nitrogen, element.oxygen]:
        if el == element.hydrogen:
            print(el, el.mass + delta_H)
        else:
            print(el, '1H', el.mass - delta_H, '2H', el.mass - 2*delta_H, '3H', el.mass - 3*delta_H)
    
    h = element.hydrogen
    h_bonds = [] #haha it is but it isn't you know?
    
    for bond in top.bonds():
        is_h = [h == atom.element for atom in [bond.atom1, bond.atom2]]
        if True in is_h: #Assume that there is at most one hydrogen per bond, thus only one element is ever true
            #Obtain Atoms
            h_atom = [bond.atom1, bond.atom2][is_h.index(True)]
            heavy_atom = [bond.atom1, bond.atom2][is_h.index(False)]
            #Indices
            h_ind = h_atom.index
            heavy_ind = heavy_atom.index
            #Masses
            
            h_mass = sys.getParticleMass(h_ind)
            heavy_mass = sys.getParticleMass(heavy_ind)
            #New Mass
            new_h_mass = h_mass + delta_H
            new_heavy_mass = heavy_mass - delta_H
            #Record new masses
            _ = sys.setParticleMass(h_ind, new_h_mass)
            _ = sys.setParticleMass(heavy_ind, new_heavy_mass)
    
    #Get some random atoms
    #print(sys.getParticleMass(420), sys.getParticleMass(69), sys.getParticleMass(649), sys.getParticleMass(469), sys.getParticleMass(690))
    
    
    input_sys = os.path.join(input_dir, name+'_sys_T.xml')
    with open(input_sys, 'w') as f:
        f.write(XmlSerializer.serialize(sys))
    
    
    # Run rep exchange
    market = FM(input_pdb=input_pdb, 
                input_system=input_sys, 
                input_state=input_state, 
                T_min=300, T_max=310,
                n_replicates=20,
                sele_str=None)
    
    
    # try:
    market.run(iter_length=iter_length,
               dt=3.0,
               sim_length=sub_sim_length,
               convergence_thresh=None,
               resSeqs=None,
               total_sim_time=total_sim_time,
               output_dir=output_dir)
    #     print('#################################################################################################################')
    #     print(f'#####################SUCCESS!############{i=}##########################################################')
    #     print('#################################################################################################################')
    # except:
    #     print('#################################################################################################################')
    #     print(f'#####################FAILURE!############{i=}##########################################################')
    #     print('#################################################################################################################')
    


#################################################################################################################
#################################i=np.float64(1.0)##########################################################
#################################################################################################################
Expect Masses
<Element hydrogen> 2.015894 Da
<Element carbon> 1H 11.002833 Da 2H 9.994886000000001 Da 3H 8.986939 Da
<Element nitrogen> 1H 12.998773 Da 2H 11.990826 Da 3H 10.982879 Da
<Element oxygen> 1H 14.991483 Da 2H 13.983536 Da 3H 12.975589 Da
03/04/2026 00:14:12//Welcome to FultonMarket.
03/04/2026 00:14:13//Found input_pdb: /expanse/lustre/projects/iit122/josephdb/KOR_Luis/Luis_Data/RS1125.pdb
03/04/2026 00:14:14//Found input_system: /expanse/lustre/projects/iit122/josephdb/KOR_Luis/Luis_Data/RS1125_sys_T.xml
03/04/2026 00:14:38//Found input_state: /expanse/lustre/projects/iit122/josephdb/KOR_Luis/Luis_Data/RS1125_state.xml
03/04/2026 00:14:38//Fou

AttributeError: module 'openmm.unit' has no attribute 'Kelvin'